# 02 - Qualidade de Dados e Camada Silver

Este notebook realiza a análise de qualidade e as transformações necessárias para construção da camada Silver do pipeline.

A partir das tabelas Bronze, serão avaliados aspectos de completude, consistência, unicidade, acurácia e presença de outliers.

Em seguida, serão realizadas transformações como padronização de tipos, tratamento de valores ausentes, validação de domínios e criação de atributos derivados necessários às análises posteriores.

In [0]:
df_vra = spark.table("workspace.mvp_sprint_3_anac.bronze_vra")
df_aerodromos = spark.table("workspace.mvp_sprint_3_anac.bronze_aerodromos")

print("VRA:", df_vra.count())
print("Aeródromos:", df_aerodromos.count())

VRA: 591447
Aeródromos: 6881


In [0]:
df_vra.printSchema()

root
 |-- sigla_icao_empresa_aerea: string (nullable = true)
 |-- empresa_aerea: string (nullable = true)
 |-- numero_voo: string (nullable = true)
 |-- codigo_di: string (nullable = true)
 |-- codigo_tipo_linha: string (nullable = true)
 |-- modelo_equipamento: string (nullable = true)
 |-- numero_de_assentos: string (nullable = true)
 |-- sigla_icao_aeroporto_origem: string (nullable = true)
 |-- descricao_aeroporto_origem: string (nullable = true)
 |-- partida_prevista: string (nullable = true)
 |-- partida_real: string (nullable = true)
 |-- sigla_icao_aeroporto_destino: string (nullable = true)
 |-- descricao_aeroporto_destino: string (nullable = true)
 |-- chegada_prevista: string (nullable = true)
 |-- chegada_real: string (nullable = true)
 |-- situacao_voo: string (nullable = true)
 |-- justificativa: string (nullable = true)
 |-- referencia: string (nullable = true)
 |-- situacao_partida: string (nullable = true)
 |-- situacao_chegada: string (nullable = true)
 |-- codeshare:

In [0]:
df_aerodromos.printSchema()

root
 |-- id_do_aerodromo: string (nullable = true)
 |-- ciad: string (nullable = true)
 |-- codigo_oaci: string (nullable = true)
 |-- id_to_tipo_de_uso: string (nullable = true)
 |-- tipo_de_uso: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- municipio_servido: string (nullable = true)
 |-- uf_servido: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- altitude: string (nullable = true)
 |-- operacao_diurna: string (nullable = true)
 |-- operacao_noturna: string (nullable = true)
 |-- id_da_pista_1: string (nullable = true)
 |-- designacao_pista_1: string (nullable = true)
 |-- comprimento_pista_1: string (nullable = true)
 |-- largura_pista_1: string (nullable = true)
 |-- resistencia_pista_1: string (nullable = true)
 |-- superficie_pista_1: string (nullable = true)
 |-- aeronave_critica_pista_1

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when, trim

def resumo_nulos(df):
    total = df.count()

    expressoes = [
        spark_sum(
            when(
                col(c).isNull() | (trim(col(c).cast("string")) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]

    resultado = df.agg(*expressoes)

    return total, resultado

In [0]:
total_vra, nulos_vra = resumo_nulos(df_vra)

print(f"Total de registros VRA: {total_vra}")
display(nulos_vra)

Total de registros VRA: 591447


sigla_icao_empresa_aerea,empresa_aerea,numero_voo,codigo_di,codigo_tipo_linha,modelo_equipamento,numero_de_assentos,sigla_icao_aeroporto_origem,descricao_aeroporto_origem,partida_prevista,partida_real,sigla_icao_aeroporto_destino,descricao_aeroporto_destino,chegada_prevista,chegada_real,situacao_voo,justificativa,referencia,situacao_partida,situacao_chegada,codeshare,arquivo_origem,data_ingestao
0,0,0,0,0,0,0,0,0,16937,18179,0,0,16937,18179,0,591447,0,35116,35116,309558,0,0


In [0]:
total_aerodromos, nulos_aerodromos = resumo_nulos(df_aerodromos)

print(f"Total de registros Aeródromos: {total_aerodromos}")
display(nulos_aerodromos)

Total de registros Aeródromos: 6881


id_do_aerodromo,ciad,codigo_oaci,id_to_tipo_de_uso,tipo_de_uso,nome,municipio,uf,pais,municipio_servido,uf_servido,latitude,longitude,altitude,operacao_diurna,operacao_noturna,id_da_pista_1,designacao_pista_1,comprimento_pista_1,largura_pista_1,resistencia_pista_1,superficie_pista_1,aeronave_critica_pista_1,tipo_operacao_por_cabeceira_pista_1,luzes_de_eixo_de_pista_pista_1,luzes_de_borda_de_pista_pista_1,id_da_pista_2,designacao_pista_2,comprimento_pista_2,largura_pista_2,resistencia_pista_2,superficie_pista_2,aeronave_critica_pista_2,tipo_operacao_por_cabeceira_pista_2,luzes_de_eixo_de_pista_pista_2,luzes_de_borda_de_pista_pista_2,tipo_de_cadastro,situacao,classe_rbac_153,perfil_operacional_rbac_153,classe_rbac_107,tipo_de_operacao_internacional_nomenclatura_oaci,tipo_de_operacao_internacional_conforme_aip_brasil,certificacao_operacional,portaria_de_certificacao,validade_do_cadastro,portaria_de_cadastro,restricao,referencia_cautelar,amazonia_legal,arquivo_origem,data_ingestao
0,22,2878,47,47,1,251,251,229,6320,6320,205,205,0,3770,3841,3208,3208,0,0,3208,3208,6829,0,3775,3701,6851,6851,6851,6851,6851,6851,6871,6851,6865,6856,0,0,6366,6463,6366,6828,6859,0,6837,1929,1931,6778,6778,0,0,0


In [0]:
def tabela_completude(df):
    total = df.count()

    dados = []

    for c in df.columns:
        ausentes = (
            df.filter(
                col(c).isNull() |
                (trim(col(c).cast("string")) == "")
            )
            .count()
        )

        percentual = (ausentes / total) * 100

        dados.append(
            (c, total, ausentes, round(percentual, 2))
        )

    return spark.createDataFrame(
        dados,
        ["coluna", "total_registros", "valores_ausentes", "percentual_ausente"]
    )

In [0]:
completude_vra = tabela_completude(df_vra)

display(
    completude_vra.orderBy(
        col("percentual_ausente").desc()
    )
)

coluna,total_registros,valores_ausentes,percentual_ausente
justificativa,591447,591447,100.0
codeshare,591447,309558,52.34
situacao_partida,591447,35116,5.94
situacao_chegada,591447,35116,5.94
partida_real,591447,18179,3.07
chegada_real,591447,18179,3.07
partida_prevista,591447,16937,2.86
chegada_prevista,591447,16937,2.86
sigla_icao_empresa_aerea,591447,0,0.0
empresa_aerea,591447,0,0.0


In [0]:
completude_aerodromos = tabela_completude(df_aerodromos)

display(
    completude_aerodromos.orderBy(
        col("percentual_ausente").desc()
    )
)

coluna,total_registros,valores_ausentes,percentual_ausente
aeronave_critica_pista_2,6881,6871,99.85
luzes_de_eixo_de_pista_pista_2,6881,6865,99.77
tipo_de_operacao_internacional_conforme_aip_brasil,6881,6859,99.68
luzes_de_borda_de_pista_pista_2,6881,6856,99.64
id_da_pista_2,6881,6851,99.56
designacao_pista_2,6881,6851,99.56
comprimento_pista_2,6881,6851,99.56
largura_pista_2,6881,6851,99.56
resistencia_pista_2,6881,6851,99.56
superficie_pista_2,6881,6851,99.56


In [0]:
total_vra = df_vra.count()
distintos_vra = df_vra.drop(
    "arquivo_origem",
    "data_ingestao"
).distinct().count()

duplicados_vra = total_vra - distintos_vra

print(f"Total VRA: {total_vra}")
print(f"Registros distintos VRA: {distintos_vra}")
print(f"Registros duplicados VRA: {duplicados_vra}")

Total VRA: 591447
Registros distintos VRA: 591431
Registros duplicados VRA: 16


In [0]:
total_aerodromos = df_aerodromos.count()
distintos_aerodromos = df_aerodromos.drop(
    "arquivo_origem",
    "data_ingestao"
).distinct().count()

duplicados_aerodromos = total_aerodromos - distintos_aerodromos

print(f"Total Aeródromos: {total_aerodromos}")
print(f"Registros distintos Aeródromos: {distintos_aerodromos}")
print(f"Registros duplicados Aeródromos: {duplicados_aerodromos}")

Total Aeródromos: 6881
Registros distintos Aeródromos: 6873
Registros duplicados Aeródromos: 8


In [0]:
from pyspark.sql.functions import count

duplicados_chave_vra = (
    df_vra
    .groupBy(
        "sigla_icao_empresa_aerea",
        "numero_voo",
        "partida_prevista",
        "sigla_icao_aeroporto_origem",
        "sigla_icao_aeroporto_destino"
    )
    .agg(count("*").alias("quantidade"))
    .filter(col("quantidade") > 1)
    .orderBy(col("quantidade").desc())
)

display(duplicados_chave_vra)

sigla_icao_empresa_aerea,numero_voo,partida_prevista,sigla_icao_aeroporto_origem,sigla_icao_aeroporto_destino,quantidade
LAN,Z0406,null,SCEL,SUMU,87
LAN,Z0407,null,SUMU,SCEL,86
ARG,1233,null,SBPA,SABE,76
JAT,Z0720,null,SCEL,SAME,75
JAT,Z0721,null,SAME,SCEL,74
ACN,Z5900,null,SBJD,SBJD,66
ARG,1269,null,SBGL,SABE,62
LAN,Z0628,null,SCEL,SAEZ,62
LAN,Z0631,null,SABE,SCEL,60
JES,Z3796,null,SABE,SGAS,60


In [0]:
duplicados_chave_aerodromos = (
    df_aerodromos
    .groupBy("id_do_aerodromo")
    .agg(count("*").alias("quantidade"))
    .filter(col("quantidade") > 1)
    .orderBy(col("quantidade").desc())
)

display(duplicados_chave_aerodromos)

id_do_aerodromo,quantidade
79,4
580,2
486,2
433,2
72,2
2,2


In [0]:
duplicados_chave_vra_ref = (
    df_vra
    .groupBy(
        "sigla_icao_empresa_aerea",
        "numero_voo",
        "referencia",
        "sigla_icao_aeroporto_origem",
        "sigla_icao_aeroporto_destino"
    )
    .agg(count("*").alias("quantidade"))
    .filter(col("quantidade") > 1)
    .orderBy(col("quantidade").desc())
)

display(duplicados_chave_vra_ref)

sigla_icao_empresa_aerea,numero_voo,referencia,sigla_icao_aeroporto_origem,sigla_icao_aeroporto_destino,quantidade
ACA,0090,14/01/2026 00:00:00,CYYZ,SBGR,4
ACA,0090,11/03/2026 00:00:00,CYYZ,SBGR,4
ACA,0090,25/02/2026 00:00:00,CYYZ,SBGR,4
ACA,0090,07/01/2026 00:00:00,CYYZ,SBGR,4
ACA,0090,18/03/2026 00:00:00,CYYZ,SBGR,4
ACA,0090,09/03/2026 00:00:00,CYYZ,SBGR,4
AAL,0995,21/05/2026 00:00:00,KMIA,SBGR,4
ACA,0090,25/03/2026 00:00:00,CYYZ,SBGR,4
ACA,0090,28/03/2026 00:00:00,CYYZ,SBGR,4
ACA,0090,23/03/2026 00:00:00,CYYZ,SBGR,4


In [0]:
ids_aerodromos_duplicados = (
    duplicados_chave_aerodromos
    .select("id_do_aerodromo")
)

aerodromos_para_inspecao = (
    df_aerodromos
    .join(
        ids_aerodromos_duplicados,
        on="id_do_aerodromo",
        how="inner"
    )
    .orderBy("id_do_aerodromo")
)

display(aerodromos_para_inspecao)

id_do_aerodromo,ciad,codigo_oaci,id_to_tipo_de_uso,tipo_de_uso,nome,municipio,uf,pais,municipio_servido,uf_servido,latitude,longitude,altitude,operacao_diurna,operacao_noturna,id_da_pista_1,designacao_pista_1,comprimento_pista_1,largura_pista_1,resistencia_pista_1,superficie_pista_1,aeronave_critica_pista_1,tipo_operacao_por_cabeceira_pista_1,luzes_de_eixo_de_pista_pista_1,luzes_de_borda_de_pista_pista_1,id_da_pista_2,designacao_pista_2,comprimento_pista_2,largura_pista_2,resistencia_pista_2,superficie_pista_2,aeronave_critica_pista_2,tipo_operacao_por_cabeceira_pista_2,luzes_de_eixo_de_pista_pista_2,luzes_de_borda_de_pista_pista_2,tipo_de_cadastro,situacao,classe_rbac_153,perfil_operacional_rbac_153,classe_rbac_107,tipo_de_operacao_internacional_nomenclatura_oaci,tipo_de_operacao_internacional_conforme_aip_brasil,certificacao_operacional,portaria_de_certificacao,validade_do_cadastro,portaria_de_cadastro,restricao,referencia_cautelar,amazonia_legal,arquivo_origem,data_ingestao
2,AC0002,SBCZ,2,Público,Cruzeiro do Sul,CRUZEIRO DO SUL,AC,Brasil,Cruzeiro do Sul,AC,"""07°35'58,0""""S""","""072°46'10,0""""W""",194 m,VFR / IFR,VFR / IFR,781,10/28,2400 m,45 m,32/F/A/X/T,Asfalto,null,10 - IFR Não Precisão; 28 - IFR Não Precisão,null,1,null,null,null,null,null,null,null,null,null,null,Aeródromo em Terras Brasileiras,Cadastrado,1,RBAC 121,AP-1,Público Táxi Aéreo e Aviação Geral (RNS),Autorização válida por 03 dias a partir de 05/12/2019.,não,null,18/11/2032,https://pergamum.anac.gov.br/arquivos/PA2022-9640.pdf,,null,sim,pda_aerodromos_publicos_caracteristicas_gerais.csv,2026-09-15T01:49:51.850Z
2,AC0002,SBCZ,2,Público,Cruzeiro do Sul,CRUZEIRO DO SUL,AC,Brasil,Cruzeiro do Sul,AC,"""07°35'58,0""""S""","""072°46'10,0""""W""",194 m,VFR / IFR,VFR / IFR,781,10/28,2400 m,45 m,32/F/A/X/T,Asfalto,null,10 - IFR Não Precisão; 28 - IFR Não Precisão,null,1,null,null,null,null,null,null,null,null,null,null,Aeródromo em Terras Brasileiras,Cadastrado,1,RBAC 121,AP-1,Público Táxi Aéreo e Aviação Geral (RNS),Autorização válida por 03 dias a partir de 05/12/2019.,não,null,18/11/2032,https://pergamum.anac.gov.br/arquivos/PA2022-9640.pdf,,null,sim,pda_aerodromos_publicos_caracteristicas_gerais.csv,2026-09-15T01:49:51.850Z
433,PA0002,SBSN,2,Público,Maestro Wilson Fonseca,SANTARÉM,PA,Brasil,Santarém,PA,"""02°25'29,0""""S""","""054°47'09,0""""W""",60 m,VFR / IFR,VFR / IFR,1961,10/28,2400 m,45 m,48/F/A/W/T,Asfalto,null,10 - IFR Não Precisão; 28 - IFR Não Precisão,null,1,null,null,null,null,null,null,null,null,null,null,Aeródromo em Terras Brasileiras,Cadastrado,2,null,AP-1,Público Regular ou Charter exclusivamente de voos alternados (AS),INTL PAX/CARGA,sim,https://pergamum.anac.gov.br/arquivos/PA2021-5751.pdf,23/09/2030,https://pergamum.anac.gov.br/arquivos/PA2020-2416.pdf,,null,sim,pda_aerodromos_publicos_caracteristicas_gerais.csv,2026-09-15T01:49:51.850Z
433,PA0002,SBSN,2,Público,Maestro Wilson Fonseca,SANTARÉM,PA,Brasil,Santarém,PA,"""02°25'29,0""""S""","""054°47'09,0""""W""",60 m,VFR / IFR,VFR / IFR,1961,10/28,2400 m,45 m,48/F/A/W/T,Asfalto,null,10 - IFR Não Precisão; 28 - IFR Não Precisão,null,1,null,null,null,null,null,null,null,null,null,null,Aeródromo em Terras Brasileiras,Cadastrado,2,null,AP-1,Público Regular ou Charter exclusivamente de voos alternados (AS),INTL PAX/CARGA,sim,https://pergamum.anac.gov.br/arquivos/PA2021-5751.pdf,23/09/2030,https://pergamum.anac.gov.br/arquivos/PA2020-2416.pdf,,null,sim,pda_aerodromos_publicos_caracteristicas_gerais.csv,2026-09-15T01:49:51.850Z
486,RO0001,SBPV,2,Público,Governador Jorge Teixeira de Oliveira,PORTO VELHO,RO,Brasil,Porto Velho,RO,"""08°42'49,0""""S""","""063°54'10,0""""W""",90 m,null,null,1922,01/19,2400 m,45 m,41/F/B/X/T,Asfalto,null,01 - ; 19 -,null,1,null,null,null,null,null,null,null,null,null,null,Aeródromo em Terras Brasileiras,Cadastrado,2,null,AP-2,Público Regular ou Charter exclusivamente de voos alternados (AS),INTL PAX/CARGA,não,null,09/02/2032,https://pergamum.anac.gov.br/arquivos/PA2022-7087

In [0]:
exemplo_duplicado_vra = (
    df_vra
    .filter(
        (col("sigla_icao_empresa_aerea") == "ACA") &
        (col("numero_voo") == "D090") &
        (col("referencia") == "14/01/2026 00:00:00") &
        (col("sigla_icao_aeroporto_origem") == "CYYZ") &
        (col("sigla_icao_aeroporto_destino") == "SBGR")
    )
)

display(exemplo_duplicado_vra)

sigla_icao_empresa_aerea,empresa_aerea,numero_voo,codigo_di,codigo_tipo_linha,modelo_equipamento,numero_de_assentos,sigla_icao_aeroporto_origem,descricao_aeroporto_origem,partida_prevista,partida_real,sigla_icao_aeroporto_destino,descricao_aeroporto_destino,chegada_prevista,chegada_real,situacao_voo,justificativa,referencia,situacao_partida,situacao_chegada,codeshare,arquivo_origem,data_ingestao


In [0]:
chave_exemplo = (
    duplicados_chave_vra_ref
    .limit(1)
    .select(
        "sigla_icao_empresa_aerea",
        "numero_voo",
        "referencia",
        "sigla_icao_aeroporto_origem",
        "sigla_icao_aeroporto_destino"
    )
)

exemplo_duplicado_vra = (
    df_vra
    .join(
        chave_exemplo,
        on=[
            "sigla_icao_empresa_aerea",
            "numero_voo",
            "referencia",
            "sigla_icao_aeroporto_origem",
            "sigla_icao_aeroporto_destino"
        ],
        how="inner"
    )
)

display(exemplo_duplicado_vra)

sigla_icao_empresa_aerea,numero_voo,referencia,sigla_icao_aeroporto_origem,sigla_icao_aeroporto_destino,empresa_aerea,codigo_di,codigo_tipo_linha,modelo_equipamento,numero_de_assentos,descricao_aeroporto_origem,partida_prevista,partida_real,descricao_aeroporto_destino,chegada_prevista,chegada_real,situacao_voo,justificativa,situacao_partida,situacao_chegada,codeshare,arquivo_origem,data_ingestao
ACA,0090,07/01/2026 00:00:00,CYYZ,SBGR,AIR CANADA,0,I,B789,298,TORONTO PEARSON INTERNATIONAL AIRPORT - TORONTO - CANADÁ,07/01/2026 23:00,07/01/2026 02:02,GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO - GUARULHOS - SP - BRASIL,08/01/2026 08:55,07/01/2026 12:13,REALIZADO,null,Antecipado,Antecipado,"AZU/7800, GLO/6804",VRA_2026_01.csv,2026-09-15T01:49:44.888Z
ACA,0090,07/01/2026 00:00:00,CYYZ,SBGR,AIR CANADA,0,I,B789,298,TORONTO PEARSON INTERNATIONAL AIRPORT - TORONTO - CANADÁ,07/01/2026 00:35,07/01/2026 23:03,GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO - GUARULHOS - SP - BRASIL,07/01/2026 10:35,08/01/2026 08:57,REALIZADO,null,Atraso > 240,Atraso > 240,"AZU/7800, GLO/6804",VRA_2026_01.csv,2026-09-15T01:49:44.888Z
ACA,0090,07/01/2026 00:00:00,CYYZ,SBGR,AIR CANADA,0,I,B789,298,TORONTO PEARSON INTERNATIONAL AIRPORT - TORONTO - CANADÁ,07/01/2026 00:35,07/01/2026 02:02,GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO - GUARULHOS - SP - BRASIL,07/01/2026 10:35,07/01/2026 12:13,REALIZADO,null,Atraso 60-120,Atraso 60-120,"AZU/7800, GLO/6804",VRA_2026_01.csv,2026-09-15T01:49:44.888Z
ACA,0090,07/01/2026 00:00:00,CYYZ,SBGR,AIR CANADA,0,I,B789,298,TORONTO PEARSON INTERNATIONAL AIRPORT - TORONTO - CANADÁ,07/01/2026 23:00,07/01/2026 23:03,GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO - GUARULHOS - SP - BRASIL,08/01/2026 08:55,08/01/2026 08:57,REALIZADO,null,Pontual,Pontual,"AZU/7800, GLO/6804",VRA_2026_01.csv,2026-09-15T01:49:44.888Z


In [0]:
pdf_exemplo = (
    exemplo_duplicado_vra
    .drop("arquivo_origem", "data_ingestao")
    .toPandas()
)

colunas_variaveis = [
    coluna
    for coluna in pdf_exemplo.columns
    if pdf_exemplo[coluna].astype(str).nunique(dropna=False) > 1
]

print("Colunas que diferem entre os 4 registros:")
for coluna in colunas_variaveis:
    print("-", coluna)

display(pdf_exemplo[colunas_variaveis])

Colunas que diferem entre os 4 registros:
- partida_prevista
- partida_real
- chegada_prevista
- chegada_real
- situacao_partida
- situacao_chegada


partida_prevista,partida_real,chegada_prevista,chegada_real,situacao_partida,situacao_chegada
07/01/2026 23:00,07/01/2026 02:02,08/01/2026 08:55,07/01/2026 12:13,Antecipado,Antecipado
07/01/2026 00:35,07/01/2026 23:03,07/01/2026 10:35,08/01/2026 08:57,Atraso > 240,Atraso > 240
07/01/2026 00:35,07/01/2026 02:02,07/01/2026 10:35,07/01/2026 12:13,Atraso 60-120,Atraso 60-120
07/01/2026 23:00,07/01/2026 23:03,08/01/2026 08:55,08/01/2026 08:57,Pontual,Pontual


In [0]:
from pyspark.sql.functions import count

for coluna in [
    "situacao_voo",
    "situacao_partida",
    "situacao_chegada",
    "codigo_tipo_linha"
]:
    print(f"\nValores de {coluna}:")
    
    display(
        df_vra
        .groupBy(coluna)
        .agg(count("*").alias("quantidade"))
        .orderBy(col("quantidade").desc())
    )


Valores de situacao_voo:


situacao_voo,quantidade
REALIZADO,573268
CANCELADO,18179



Valores de situacao_partida:


situacao_partida,quantidade
Antecipado,304091
Pontual,202732
null,35116
Atraso 30-60,28396
Atraso 60-120,13064
Atraso 120-240,5447
Atraso > 240,2601



Valores de situacao_chegada:


situacao_chegada,quantidade
Antecipado,329417
Pontual,176151
null,35116
Atraso 30-60,29654
Atraso 60-120,13227
Atraso 120-240,5356
Atraso > 240,2526



Valores de codigo_tipo_linha:


codigo_tipo_linha,quantidade
N,466037
I,104195
G,13155
C,8060


In [0]:
aeroportos_cadastrados = (
    df_aerodromos
    .select("codigo_oaci")
    .filter(col("codigo_oaci").isNotNull())
    .distinct()
)

origens_sem_cadastro = (
    df_vra
    .select(
        col("sigla_icao_aeroporto_origem").alias("codigo_oaci")
    )
    .distinct()
    .join(
        aeroportos_cadastrados,
        on="codigo_oaci",
        how="left_anti"
    )
)

destinos_sem_cadastro = (
    df_vra
    .select(
        col("sigla_icao_aeroporto_destino").alias("codigo_oaci")
    )
    .distinct()
    .join(
        aeroportos_cadastrados,
        on="codigo_oaci",
        how="left_anti"
    )
)

print("Códigos de origem sem correspondência no cadastro:")
display(origens_sem_cadastro)

print("Códigos de destino sem correspondência no cadastro:")
display(destinos_sem_cadastro)

Códigos de origem sem correspondência no cadastro:


codigo_oaci
SOCA
VTBS
LFPG
SAMU
GMMN
EHAM
KAUS
MPPA
SVPR
GOBD


Códigos de destino sem correspondência no cadastro:


codigo_oaci
SOCA
LHBP
VTBS
SLLP
LFPG
GMMN
EHAM
GOBD
KAUS
SVPR


In [0]:
from pyspark.sql.functions import to_timestamp, sum as spark_sum, when

formato_data_hora = "dd/MM/yyyy HH:mm"

checagem_datas_vra = (
    df_vra
    .select(
        "*",
        to_timestamp(col("partida_prevista"), formato_data_hora).alias("_partida_prevista_ts"),
        to_timestamp(col("partida_real"), formato_data_hora).alias("_partida_real_ts"),
        to_timestamp(col("chegada_prevista"), formato_data_hora).alias("_chegada_prevista_ts"),
        to_timestamp(col("chegada_real"), formato_data_hora).alias("_chegada_real_ts"),
        to_timestamp(col("referencia"), "dd/MM/yyyy HH:mm:ss").alias("_referencia_ts")
    )
)

In [0]:
display(
    checagem_datas_vra.select(
        spark_sum(
            when(
                col("partida_prevista").isNotNull() &
                col("_partida_prevista_ts").isNull(),
                1
            ).otherwise(0)
        ).alias("partida_prevista_invalida"),

        spark_sum(
            when(
                col("partida_real").isNotNull() &
                col("_partida_real_ts").isNull(),
                1
            ).otherwise(0)
        ).alias("partida_real_invalida"),

        spark_sum(
            when(
                col("chegada_prevista").isNotNull() &
                col("_chegada_prevista_ts").isNull(),
                1
            ).otherwise(0)
        ).alias("chegada_prevista_invalida"),

        spark_sum(
            when(
                col("chegada_real").isNotNull() &
                col("_chegada_real_ts").isNull(),
                1
            ).otherwise(0)
        ).alias("chegada_real_invalida"),

        spark_sum(
            when(
                col("referencia").isNotNull() &
                col("_referencia_ts").isNull(),
                1
            ).otherwise(0)
        ).alias("referencia_invalida")
    )
)

partida_prevista_invalida,partida_real_invalida,chegada_prevista_invalida,chegada_real_invalida,referencia_invalida
0,0,0,0,0


In [0]:
from pyspark.sql.functions import min as spark_min, max as spark_max, avg

display(
    df_vra
    .select(
        col("numero_de_assentos").cast("int").alias("numero_de_assentos")
    )
    .agg(
        spark_min("numero_de_assentos").alias("minimo"),
        spark_max("numero_de_assentos").alias("maximo"),
        avg("numero_de_assentos").alias("media")
    )
)

minimo,maximo,media
0,515,170.5283415081994


In [0]:
from pyspark.sql.functions import unix_timestamp

df_atrasos = (
    checagem_datas_vra
    .withColumn(
        "atraso_partida_min",
        (
            unix_timestamp(col("_partida_real_ts")) -
            unix_timestamp(col("_partida_prevista_ts"))
        ) / 60
    )
    .withColumn(
        "atraso_chegada_min",
        (
            unix_timestamp(col("_chegada_real_ts")) -
            unix_timestamp(col("_chegada_prevista_ts"))
        ) / 60
    )
)

In [0]:
display(
    df_atrasos
    .agg(
        spark_min("atraso_partida_min").alias("min_atraso_partida"),
        spark_max("atraso_partida_min").alias("max_atraso_partida"),
        avg("atraso_partida_min").alias("media_atraso_partida"),

        spark_min("atraso_chegada_min").alias("min_atraso_chegada"),
        spark_max("atraso_chegada_min").alias("max_atraso_chegada"),
        avg("atraso_chegada_min").alias("media_atraso_chegada")
    )
)

min_atraso_partida,max_atraso_partida,media_atraso_partida,min_atraso_chegada,max_atraso_chegada,media_atraso_chegada
-43057.0,44855.0,7.305871864052156,-43082.0,44854.0,3.9681772182387824


In [0]:
display(
    df_atrasos
    .filter(
        (col("atraso_partida_min") < -180) |
        (col("atraso_partida_min") > 600) |
        (col("atraso_chegada_min") < -180) |
        (col("atraso_chegada_min") > 600)
    )
    .select(
        "sigla_icao_empresa_aerea",
        "numero_voo",
        "referencia",
        "sigla_icao_aeroporto_origem",
        "sigla_icao_aeroporto_destino",
        "partida_prevista",
        "partida_real",
        "chegada_prevista",
        "chegada_real",
        "atraso_partida_min",
        "atraso_chegada_min",
        "situacao_voo"
    )
    .orderBy(col("atraso_partida_min").desc_nulls_last())
)

sigla_icao_empresa_aerea,numero_voo,referencia,sigla_icao_aeroporto_origem,sigla_icao_aeroporto_destino,partida_prevista,partida_real,chegada_prevista,chegada_real,atraso_partida_min,atraso_chegada_min,situacao_voo
GLO,1182,29/03/2026 00:00:00,SBGR,SBFI,29/03/2026 23:00,30/04/2026 02:35,30/03/2026 00:50,30/04/2026 04:24,44855.0,44854.0,REALIZADO
GTI,0064,11/02/2026 00:00:00,SBKP,SPJC,11/02/2026 16:10,15/02/2026 09:16,11/02/2026 21:10,15/02/2026 13:59,5346.0,5329.0,REALIZADO
GTI,8050,27/01/2026 00:00:00,KMIA,SBGR,27/01/2026 20:20,30/01/2026 14:55,28/01/2026 04:20,30/01/2026 23:35,3995.0,4035.0,REALIZADO
UAE,9941,23/06/2026 00:00:00,SBGR,SAEZ,24/06/2026 08:25,27/06/2026 01:19,24/06/2026 11:15,27/06/2026 04:15,3894.0,3900.0,REALIZADO
GTI,0058,04/07/2026 00:00:00,SBKP,SEQM,04/07/2026 19:05,07/07/2026 08:51,05/07/2026 00:35,07/07/2026 14:53,3706.0,3738.0,REALIZADO
GTI,0057,04/07/2026 00:00:00,KMIA,SBKP,04/07/2026 09:05,06/07/2026 21:39,04/07/2026 17:05,07/07/2026 05:25,3634.0,3620.0,REALIZADO
GTI,0061,20/02/2026 00:00:00,SBKP,SCEL,20/02/2026 16:10,22/02/2026 23:28,20/02/2026 20:40,23/02/2026 03:15,3318.0,3275.0,REALIZADO
GTI,0032,17/02/2026 00:00:00,SBKP,KMIA,17/02/2026 19:00,19/02/2026 23:50,18/02/2026 03:00,20/02/2026 08:03,3170.0,3183.0,REALIZADO
GTI,0045,30/07/2026 00:00:00,KMIA,SBGL,30/07/2026 11:20,01/08/2026 12:07,30/07/2026 19:20,01/08/2026 20:36,2927.0,2956.0,REALIZADO
AEA,0058,25/05/2026 00:00:00,SBGR,LEMD,25/05/2026 13:50,27/05/2026 13:35,26/05/2026 00:00,27/05/2026 23:11,2865.0,2831.0,REALIZADO


In [0]:
display(
    df_atrasos
    .filter(
        (col("atraso_partida_min") < -180) |
        (col("atraso_partida_min") > 600)
    )
    .select(
        "sigla_icao_empresa_aerea",
        "numero_voo",
        "referencia",
        "partida_prevista",
        "partida_real",
        "atraso_partida_min",
        "situacao_partida",
        "situacao_voo"
    )
    .orderBy(col("atraso_partida_min").desc())
)

sigla_icao_empresa_aerea,numero_voo,referencia,partida_prevista,partida_real,atraso_partida_min,situacao_partida,situacao_voo
GLO,1182,29/03/2026 00:00:00,29/03/2026 23:00,30/04/2026 02:35,44855.0,Atraso > 240,REALIZADO
GTI,0064,11/02/2026 00:00:00,11/02/2026 16:10,15/02/2026 09:16,5346.0,Atraso > 240,REALIZADO
GTI,8050,27/01/2026 00:00:00,27/01/2026 20:20,30/01/2026 14:55,3995.0,Atraso > 240,REALIZADO
UAE,9941,23/06/2026 00:00:00,24/06/2026 08:25,27/06/2026 01:19,3894.0,Atraso > 240,REALIZADO
GTI,0058,04/07/2026 00:00:00,04/07/2026 19:05,07/07/2026 08:51,3706.0,Atraso > 240,REALIZADO
GTI,0057,04/07/2026 00:00:00,04/07/2026 09:05,06/07/2026 21:39,3634.0,Atraso > 240,REALIZADO
GTI,0061,20/02/2026 00:00:00,20/02/2026 16:10,22/02/2026 23:28,3318.0,Atraso > 240,REALIZADO
GTI,0032,17/02/2026 00:00:00,17/02/2026 19:00,19/02/2026 23:50,3170.0,Atraso > 240,REALIZADO
GTI,0045,30/07/2026 00:00:00,30/07/2026 11:20,01/08/2026 12:07,2927.0,Atraso > 240,REALIZADO
AEA,0058,25/05/2026 00:00:00,25/05/2026 13:50,27/05/2026 13:35,2865.0,Atraso > 240,REALIZADO


In [0]:
display(
    df_atrasos
    .filter(
        (col("atraso_partida_min") < -180) |
        (col("atraso_partida_min") > 600)
    )
    .groupBy("situacao_partida")
    .agg(count("*").alias("quantidade"))
    .orderBy(col("quantidade").desc())
)

situacao_partida,quantidade
Atraso > 240,723
Antecipado,235


In [0]:
print(
    "Registros com atraso de partida extremo:",
    df_atrasos.filter(
        (col("atraso_partida_min") < -180) |
        (col("atraso_partida_min") > 600)
    ).count()
)

print(
    "Percentual sobre o VRA:",
    round(
        100 * df_atrasos.filter(
            (col("atraso_partida_min") < -180) |
            (col("atraso_partida_min") > 600)
        ).count() / df_vra.count(),
        4
    ),
    "%"
)

Registros com atraso de partida extremo: 958
Percentual sobre o VRA: 0.162 %


In [0]:
extremos_partida = df_atrasos.filter(
    (col("atraso_partida_min") < -180) |
    (col("atraso_partida_min") > 600)
)

extremos_chegada = df_atrasos.filter(
    (col("atraso_chegada_min") < -180) |
    (col("atraso_chegada_min") > 600)
)

extremos_gerais = df_atrasos.filter(
    (col("atraso_partida_min") < -180) |
    (col("atraso_partida_min") > 600) |
    (col("atraso_chegada_min") < -180) |
    (col("atraso_chegada_min") > 600)
)

print("Extremos de partida:", extremos_partida.count())
print("Extremos de chegada:", extremos_chegada.count())
print("Extremos em partida OU chegada:", extremos_gerais.count())

Extremos de partida: 958
Extremos de chegada: 955
Extremos em partida OU chegada: 982


In [0]:
from pyspark.sql.functions import (
    col,
    to_timestamp,
    when,
    hour,
    dayofweek,
    month,
    year
)

df_vra_silver = (
    df_vra
    .dropDuplicates()

    .withColumn(
        "partida_prevista_ts",
        to_timestamp(col("partida_prevista"), "dd/MM/yyyy HH:mm")
    )
    .withColumn(
        "partida_real_ts",
        to_timestamp(col("partida_real"), "dd/MM/yyyy HH:mm")
    )
    .withColumn(
        "chegada_prevista_ts",
        to_timestamp(col("chegada_prevista"), "dd/MM/yyyy HH:mm")
    )
    .withColumn(
        "chegada_real_ts",
        to_timestamp(col("chegada_real"), "dd/MM/yyyy HH:mm")
    )
    .withColumn(
        "referencia_ts",
        to_timestamp(col("referencia"), "dd/MM/yyyy HH:mm:ss")
    )
    .withColumn(
        "numero_de_assentos_int",
        col("numero_de_assentos").cast("int")
    )

    .withColumn(
        "atraso_partida_min",
        (
            unix_timestamp(col("partida_real_ts")) -
            unix_timestamp(col("partida_prevista_ts"))
        ) / 60
    )
    .withColumn(
        "atraso_chegada_min",
        (
            unix_timestamp(col("chegada_real_ts")) -
            unix_timestamp(col("chegada_prevista_ts"))
        ) / 60
    )

    .withColumn(
        "flag_atraso_30",
        when(col("atraso_partida_min").isNull(), None)
        .when(col("atraso_partida_min") > 30, 1)
        .otherwise(0)
    )
    .withColumn(
        "flag_atraso_60",
        when(col("atraso_partida_min").isNull(), None)
        .when(col("atraso_partida_min") > 60, 1)
        .otherwise(0)
    )
    .withColumn(
        "flag_cancelado",
        when(col("situacao_voo") == "CANCELADO", 1).otherwise(0)
    )
    .withColumn(
        "flag_atraso_extremo",
        when(
            col("atraso_partida_min").isNull() &
            col("atraso_chegada_min").isNull(),
            None
        )
        .when(
            (col("atraso_partida_min") < -180) |
            (col("atraso_partida_min") > 600) |
            (col("atraso_chegada_min") < -180) |
            (col("atraso_chegada_min") > 600),
            1
        )
        .otherwise(0)
    )

    .withColumn("ano", year(col("referencia_ts")))
    .withColumn("mes", month(col("referencia_ts")))
    .withColumn("dia_semana_num", dayofweek(col("referencia_ts")))
    .withColumn(
        "hora_partida_prevista",
        hour(col("partida_prevista_ts"))
    )
    .withColumn(
        "periodo_dia",
        when(col("hora_partida_prevista").isNull(), "Não informado")
        .when(
            (col("hora_partida_prevista") >= 5) &
            (col("hora_partida_prevista") < 12),
            "Manhã"
        )
        .when(
            (col("hora_partida_prevista") >= 12) &
            (col("hora_partida_prevista") < 18),
            "Tarde"
        )
        .when(
            (col("hora_partida_prevista") >= 18) &
            (col("hora_partida_prevista") < 24),
            "Noite"
        )
        .otherwise("Madrugada")
    )
)

In [0]:
from pyspark.sql.functions import expr

df_aerodromos_silver = (
    df_aerodromos
    .dropDuplicates()

    .withColumn(
        "latitude_double",
        expr("""
            CASE
                WHEN latitude IS NULL OR trim(latitude) = '' THEN NULL
                ELSE
                    (
                        try_cast(regexp_extract(latitude, '(\\\\d+)°', 1) AS DOUBLE)
                        +
                        try_cast(regexp_extract(latitude, '°(\\\\d+)''', 1) AS DOUBLE) / 60
                        +
                        try_cast(
                            replace(
                                regexp_extract(latitude, '''([\\\\d,]+)', 1),
                                ',',
                                '.'
                            )
                            AS DOUBLE
                        ) / 3600
                    )
                    *
                    CASE
                        WHEN latitude RLIKE '[SW]$' THEN -1
                        ELSE 1
                    END
            END
        """)
    )

    .withColumn(
        "longitude_double",
        expr("""
            CASE
                WHEN longitude IS NULL OR trim(longitude) = '' THEN NULL
                ELSE
                    (
                        try_cast(regexp_extract(longitude, '(\\\\d+)°', 1) AS DOUBLE)
                        +
                        try_cast(regexp_extract(longitude, '°(\\\\d+)''', 1) AS DOUBLE) / 60
                        +
                        try_cast(
                            replace(
                                regexp_extract(longitude, '''([\\\\d,]+)', 1),
                                ',',
                                '.'
                            )
                            AS DOUBLE
                        ) / 3600
                    )
                    *
                    CASE
                        WHEN longitude RLIKE '[SW]$' THEN -1
                        ELSE 1
                    END
            END
        """)
    )

    .withColumn(
        "altitude_double",
        expr("""
            try_cast(
                replace(
                    regexp_extract(altitude, '(-?[0-9,.]+)', 1),
                    ',',
                    '.'
                )
                AS DOUBLE
            )
        """)
    )
)

In [0]:
print("VRA Bronze:", df_vra.count())
print("VRA Silver:", df_vra_silver.count())

print("Aeródromos Bronze:", df_aerodromos.count())
print("Aeródromos Silver:", df_aerodromos_silver.count())

VRA Bronze: 591447
VRA Silver: 591431
Aeródromos Bronze: 6881
Aeródromos Silver: 6873


In [0]:
df_vra_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_sprint_3_anac.silver_vra")

In [0]:
df_aerodromos_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_sprint_3_anac.silver_aerodromos")

In [0]:
print(
    "silver_vra:",
    spark.table("workspace.mvp_sprint_3_anac.silver_vra").count()
)

print(
    "silver_aerodromos:",
    spark.table("workspace.mvp_sprint_3_anac.silver_aerodromos").count()
)

silver_vra: 591431
silver_aerodromos: 6873


In [0]:
display(
    spark.sql(
        "SHOW TABLES IN workspace.mvp_sprint_3_anac"
    )
)

database,tableName,isTemporary
mvp_sprint_3_anac,bronze_aerodromos,false
mvp_sprint_3_anac,bronze_vra,false
mvp_sprint_3_anac,silver_aerodromos,false
mvp_sprint_3_anac,silver_vra,false


## Resultado da camada Silver

A camada Silver foi criada a partir das tabelas Bronze após a execução das verificações de qualidade e das transformações necessárias ao pipeline.

Principais tratamentos realizados:

- remoção de duplicatas exatas;
- conversão dos campos temporais para tipos adequados;
- criação de métricas de atraso de partida e chegada;
- criação de indicadores de atraso superior a 30 e 60 minutos;
- criação de indicador de cancelamento;
- identificação de atrasos e antecipações extremos sem exclusão automática dos registros;
- criação de atributos temporais derivados, como ano, mês, dia da semana, hora prevista e período do dia;
- conversão das coordenadas dos aeródromos para formato decimal;
- tratamento tolerante de valores numéricos malformados, preservando-os como valores nulos quando a conversão não foi possível.

Após a deduplicação, a tabela `silver_vra` passou de 591.447 para 591.431 registros, enquanto `silver_aerodromos` passou de 6.881 para 6.873 registros.

Os registros operacionais extremos foram preservados, pois a análise mostrou que a própria classificação da ANAC reconhece esses casos como atrasos ou antecipações, não havendo evidência suficiente para tratá-los como erros.